In [20]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

# ----------------------------------------------------
# 1. 读取数据、归一化并定义函数
# ----------------------------------------------------

# 定义文件路径
file_path = 'input_slip_sys_fcc.txt'

try:
    # 读取数据，注意文件中没有列名
    slip_data = np.loadtxt(file_path)

    # 将数据转换为DataFrame以便于管理和计算
    slip_df = pd.DataFrame(slip_data, columns=[
        'n_x', 'n_y', 'n_z',
        'b_x', 'b_y', 'b_z'
    ])
    
    # 对滑移面法向量和Burgers矢量进行归一化
    # 计算 n 和 b 向量的模
    n_norms = np.linalg.norm(slip_df.iloc[:, 0:3].values, axis=1)
    b_norms = np.linalg.norm(slip_df.iloc[:, 3:6].values, axis=1)
    
    # 归一化
    slip_df.iloc[:, 0:3] = slip_df.iloc[:, 0:3].values / n_norms[:, np.newaxis]
    slip_df.iloc[:, 3:6] = slip_df.iloc[:, 3:6].values / b_norms[:, np.newaxis]
    
    # 为每个滑移系添加一个序号
    slip_df.index.name = 'Slip System'
    
    print("成功读取滑移系数据，并已完成归一化。共", len(slip_df), "个滑移系。")
    print("\n归一化后的数据前5行：")
    # print(slip_df.head())
    
except FileNotFoundError:
    print(f"错误：文件 '{file_path}' 未找到。请确保它存在。")
    slip_df = pd.DataFrame()

成功读取滑移系数据，并已完成归一化。共 12 个滑移系。

归一化后的数据前5行：


In [21]:
phi1_deg = 0.0
Phi_deg = 0.0
phi2_deg = 0.0

# 转化为弧度
phi1_rad = np.deg2rad(phi1_deg)
Phi_rad = np.deg2rad(Phi_deg)
phi2_rad = np.deg2rad(phi2_deg)

# 拉伸轴方向 (在样本坐标系下)
# 这是一个单位向量
s_sample = np.array([0, 1, 0]) # 例如：沿Z轴拉伸
s_sample = s_sample / np.linalg.norm(s_sample)

print(f"设置的欧拉角 (Bunge): ({phi1_deg:.2f}, {Phi_deg:.2f}, {phi2_deg:.2f}) 度")
print(f"设置的拉伸方向向量: {s_sample}")

设置的欧拉角 (Bunge): (0.00, 0.00, 0.00) 度
设置的拉伸方向向量: [0. 1. 0.]


In [22]:
rotation_matrix = R.from_euler('ZXZ', [phi1_rad, Phi_rad, phi2_rad]).as_matrix()
schmid_factors = []

# 遍历每一个滑移系
for index, row in slip_df.iterrows():
    # 提取晶体坐标系下的滑移面法向量 n 和 Burgers 矢量 b
    n_crystal = np.array([row['n_x'], row['n_y'], row['n_z']])
    b_crystal = np.array([row['b_x'], row['b_y'], row['b_z']])
    
    # 将 n 和 b 从晶体坐标系转换到样本坐标系
    # 这里的旋转矩阵将向量从晶体坐标系旋转到样本坐标系
    n_sample = rotation_matrix @ n_crystal
    b_sample = rotation_matrix @ b_crystal
    
    # 计算 Schmid Factor: m = (s · n) * (s · b)
    schmid_factor = np.dot(s_sample, n_sample) * np.dot(s_sample, b_sample)
    
    schmid_factors.append(np.abs(schmid_factor)) # 通常取绝对值

# 将结果添加到DataFrame
slip_df['Schmid Factor'] = schmid_factors

# ----------------------------------------------------
# 4. 展示结果
# ----------------------------------------------------

print("所有滑移系的Schmid Factor已计算完成。")
slip_df_sorted = slip_df.sort_values(by='Schmid Factor', ascending=False)

# 打印最终结果，按Schmid Factor降序排列
slip_df

所有滑移系的Schmid Factor已计算完成。


,n_x,n_y,n_z,b_x,b_y,b_z,Schmid Factor
Slip System,,,,,,,
0,0.57735,0.57735,-0.57735,0.000000,0.707107,0.707107,0.408248
1,0.57735,0.57735,-0.57735,0.707107,0.000000,0.707107,0.000000
2,0.57735,0.57735,-0.57735,0.707107,-0.707107,0.000000,0.408248
3,0.57735,-0.57735,-0.57735,0.000000,0.707107,-0.707107,0.408248
4,0.57735,-0.57735,-0.57735,0.707107,0.000000,0.707107,0.000000
5,0.57735,-0.57735,-0.57735,0.707107,0.707107,0.000000,0.408248
6,0.57735,-0.57735,0.57735,0.000000,0.707107,0.707107,0.408248
7,0.57735,-0.57735,0.57735,0.707107,0.000000,-0.707107,0.000000
8,0.57735,-0.57735,0.57735,0.707107,0.707107,0.000000,0.408248
